In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

In [ ]:
df = pd.read_csv('../data/fully combined/wr_all_weeks.csv')
print(f'Shape: {df.shape}')
print(f'Sezone: {df["season"].min()} – {df["season"].max()}')
df.head()

In [ ]:
df.dtypes.to_frame('dtype').reset_index().rename(columns={'index': 'column'})

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

In [ ]:
df.describe().T

In [ ]:
df.groupby('season').size().rename('zapisi').reset_index()

In [ ]:
df['week'] = df['game_id'].str.split('_').str[1].astype(int)
df = df.sort_values(['receiver_player_id', 'season', 'week'], ascending=[True, True, True]).reset_index(drop=True)

print('Tjedni:', sorted(df['week'].unique()))
df.groupby('week').size().rename('zapisi').reset_index()

In [ ]:
df['weeks_since_last_game'] = (
    df.groupby(['receiver_player_id', 'season'])['week']
      .diff()
      .fillna(1)
      .clip(lower=1)
      .astype(int)
)

print(df['weeks_since_last_game'].value_counts().sort_index().head(10))
df[['receiver_player_id', 'season', 'week', 'weeks_since_last_game']].head(20)

In [ ]:
games_per_player_season = (
    df.groupby(['receiver_player_id', 'season'])
      .size()
      .rename('games_in_season')
      .reset_index()
 )

player_seasons_over_3 = games_per_player_season[games_per_player_season['games_in_season'] > 3].copy()
player_seasons_over_5 = games_per_player_season[games_per_player_season['games_in_season'] > 5].copy()

total_players = games_per_player_season['receiver_player_id'].nunique()
total_player_seasons = len(games_per_player_season)

num_players_over_3 = player_seasons_over_3['receiver_player_id'].nunique()
num_player_seasons_over_3 = len(player_seasons_over_3)

num_players_over_5 = player_seasons_over_5['receiver_player_id'].nunique()
num_player_seasons_over_5 = len(player_seasons_over_5)

pct_players_over_3 = (num_players_over_3 / total_players) * 100
pct_player_seasons_over_3 = (num_player_seasons_over_3 / total_player_seasons) * 100

pct_players_over_5 = (num_players_over_5 / total_players) * 100
pct_player_seasons_over_5 = (num_player_seasons_over_5 / total_player_seasons) * 100

print(f'Ukupno igraca: {total_players}')
print(f'Igraca sa bar jednom sezonom >3 utakmice: {num_players_over_3} ({pct_players_over_3:.2f}%)')
print(f'Igraca sa bar jednom sezonom >5 utakmica: {num_players_over_5} ({pct_players_over_5:.2f}%)')
print('---')
print(f'Ukupno igrac-sezona kombinacija: {total_player_seasons}')
print(f'Kombinacija sa >3 utakmice: {num_player_seasons_over_3} ({pct_player_seasons_over_3:.2f}%)')
print(f'Kombinacija sa >5 utakmica: {num_player_seasons_over_5} ({pct_player_seasons_over_5:.2f}%)')

comparison = pd.DataFrame({
    'metric': [
        'Ukupno igraca',
        'Igraci sa bar jednom sezonom >3',
        'Igraci sa bar jednom sezonom >5',
        'Ukupno igrac-sezona kombinacija',
        'Igrac-sezona kombinacije >3',
        'Igrac-sezona kombinacije >5'
    ],
    'value': [
        total_players,
        num_players_over_3,
        num_players_over_5,
        total_player_seasons,
        num_player_seasons_over_3,
        num_player_seasons_over_5
    ]
})

comparison

In [ ]:
# Inferred player team from matchup: offense is the opposite side of defteam.
df['player_team_inferred'] = np.where(
    (df['home_team'] == df['defteam']) & (df['away_team'] != df['defteam']),
    df['away_team'],
    np.where(
        (df['away_team'] == df['defteam']) & (df['home_team'] != df['defteam']),
        df['home_team'],
        np.nan
    )
)

prev_team = df.groupby('receiver_player_id')['player_team_inferred'].shift(1)
df['team_changed'] = (
    df['player_team_inferred'].notna()
    & prev_team.notna()
    & (df['player_team_inferred'] != prev_team)
).astype(int)

prev_season = df.groupby('receiver_player_id')['season'].shift(1)
df['is_new_season'] = (
    prev_season.notna()
    & (df['season'] != prev_season)
).astype(int)

unknown_team_rows = df['player_team_inferred'].isna().sum()
print(f'Rows with unresolved inferred team: {unknown_team_rows}')
print('team_changed distribution:')
print(df['team_changed'].value_counts(dropna=False).sort_index())
print('is_new_season distribution:')
print(df['is_new_season'].value_counts(dropna=False).sort_index())

df.loc[
    (df['team_changed'] == 1) | (df['is_new_season'] == 1),
    ['receiver_player_id', 'season', 'week', 'player_team_inferred', 'team_changed', 'is_new_season']
].head(20)